# Practical 4 — Stemming & Lemmatization

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To reduce words to their base/root form using stemming (Porter) and lemmatization (WordNet, with and without POS information), compare their outputs on irregular forms, and measure the effect on vocabulary size.

## Theory

Both stemming and lemmatization aim to collapse different inflected forms of a word ("run", "running", "ran") down to one representative form, so they're counted as the same feature instead of three separate ones.

**Stemming** (e.g. the Porter algorithm) works by applying a fixed set of suffix-stripping rules — it has no knowledge of the language beyond those rules. This makes it fast, but:
- It can produce forms that aren't real words at all (e.g. "studies" -> "studi").
- It has no concept of irregular forms — an irregular verb like "seen" or "outdid" won't be reduced to its base form, since there's no suffix pattern to strip.

**Lemmatization** (e.g. NLTK's WordNet lemmatizer) instead looks the word up against a real vocabulary/morphological database, returning an actual dictionary base form (a *lemma*). This is more linguistically correct, but:
- It needs to know the word's **part of speech** to work correctly. Without it, WordNetLemmatizer assumes every word is a **noun** — so a verb like "seen" (past participle of "see") won't get correctly reduced, because "seen" isn't a valid noun form to reduce.
- With the correct POS supplied, it can resolve irregular forms that stemming simply cannot touch.

This practical exists specifically to make that POS-dependency concrete rather than theoretical, using real words pulled from the review dataset.

## Algorithm

1. Reuse cleaned tokens from Practical 1.
2. Apply Porter stemming to a handful of words known to have irregular forms ("seen", "outdid", "given", "dragged") and inspect the output.
3. Apply lemmatization to the same words twice — once assuming every word is a noun (no POS), once using each word's actual POS tag — and compare all three outputs side by side.
4. Run stemming and POS-aware lemmatization across the whole dataset and compute vocabulary size for each, compared against the Practical 1 baseline (136).
5. Identify any words where the three methods disagree, and inspect whether the stemmed form is a real word or not.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg)
    except Exception as e:
        print(f"Skipped {pkg}: {e}")

import preprocessing
import tokenizer
import stemming_lemmatization as norm

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
df["tokens"] = df["review"].apply(lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r)))
print(f"Loaded {len(df)} reviews")


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Loaded 15 reviews


### Step 1 — Irregular forms: stemming vs lemmatization (no POS) vs lemmatization (with POS)

These four words were picked because they're all irregular or otherwise tricky: "seen" (irregular past participle of "see"), "outdid" (irregular past tense of "outdo"), "given" (irregular past participle of "give"), "dragged" (regular, but tests suffix-stripping on a doubled consonant).

In [2]:
test_words = ["seen", "outdid", "given", "dragged"]
result = norm.compare_stem_lemma(test_words)

for word in test_words:
    idx = test_words.index(word)
    print(f"{word:>10}  ->  stemmed: {result['stemmed'][idx]:<10}  "
          f"lemma (no POS): {result['lemmatized_no_pos'][idx]:<10}  "
          f"lemma (with POS): {result['lemmatized_with_pos'][idx]}")


      seen  ->  stemmed: seen        lemma (no POS): seen        lemma (with POS): see
    outdid  ->  stemmed: outdid      lemma (no POS): outdid      lemma (with POS): outdid
     given  ->  stemmed: given       lemma (no POS): given       lemma (with POS): give
   dragged  ->  stemmed: drag        lemma (no POS): dragged     lemma (with POS): drag


**Look carefully at whether the POS-aware lemmatizer actually succeeded on all four, or only some — WordNetLemmatizer's built-in exception list doesn't necessarily cover every irregular verb. Note in your Output section which words it got right and which (if any) it didn't.**

### Step 2 — Full pipeline: vocabulary size comparison

In [3]:
all_raw_tokens = df["tokens"].tolist()
all_stemmed = [norm.porter_stem(tokens) for tokens in all_raw_tokens]
all_lemmatized = [norm.lemmatize_tokens(tokens, use_pos=True) for tokens in all_raw_tokens]

def vocab_size(token_lists):
    return len(set(t for tokens in token_lists for t in tokens))

raw_vocab = vocab_size(all_raw_tokens)
stemmed_vocab = vocab_size(all_stemmed)
lemmatized_vocab = vocab_size(all_lemmatized)

print(f"Raw vocabulary (Practical 1 baseline):  {raw_vocab}")
print(f"Vocabulary after stemming:               {stemmed_vocab}")
print(f"Vocabulary after POS-aware lemmatization: {lemmatized_vocab}")


Raw vocabulary (Practical 1 baseline):  136
Vocabulary after stemming:               133
Vocabulary after POS-aware lemmatization: 130


### Step 3 — Where do stemming and lemmatization disagree?

In [4]:
disagreements = []
for i in range(len(all_raw_tokens)):
    for raw, stem, lemma in zip(all_raw_tokens[i], all_stemmed[i], all_lemmatized[i]):
        if stem != lemma and (raw, stem, lemma) not in disagreements:
            disagreements.append((raw, stem, lemma))

print(f"{len(disagreements)} words where stemmed and lemmatized forms differ:\n")
for raw, stem, lemma in sorted(set(disagreements)):
    print(f"  {raw:>15}  ->  stem: {stem:<12}  lemma: {lemma}")


38 words where stemmed and lemmatized forms differ:

       absolutely  ->  stem: absolut       lemma: absolutely
           acting  ->  stem: act           lemma: acting
        admission  ->  stem: admiss        lemma: admission
            alone  ->  stem: alon          lemma: alone
           anyone  ->  stem: anyon         lemma: anyone
         anything  ->  stem: anyth         lemma: anything
              are  ->  stem: are           lemma: be
           before  ->  stem: befor         lemma: before
          believe  ->  stem: believ        lemma: believe
        confusing  ->  stem: confus        lemma: confuse
           decade  ->  stem: decad         lemma: decade
         dialogue  ->  stem: dialogu       lemma: dialogue
            every  ->  stem: everi         lemma: every
        fantastic  ->  stem: fantast       lemma: fantastic
            given  ->  stem: given         lemma: give
              his  ->  stem: hi            lemma: his
         honestly  ->  stem: h

## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- For the four irregular test words, did POS-aware lemmatization actually resolve all of them correctly, or did any still come out wrong? What does that tell you about relying on WordNetLemmatizer alone for irregular verbs?
- How much did vocabulary size shrink under stemming vs lemmatization — which one collapsed more words together, and does that surprise you given what each method is supposed to do?
- Looking at the disagreement list, pick 2-3 examples where the stemmed form looks like it might not be a real word (over-stemming) — did you find any?
- Given the trade-offs (speed vs accuracy, real words vs non-words), which would you pick for a downstream task like sentiment classification on this dataset, and why?

### answer 
- The results show that stemming and lemmatization both reduce vocabulary size, but lemmatization performs a more linguistically accurate normalization. The original vocabulary of '136' words was reduced to 133 after stemming and further to 130 after POS-aware lemmatization, indicating that lemmatization merged more word variants into their correct base forms. Stemming often produced truncated or non-dictionary words such as "absolut", "fantast", "movi", and "terribl", whereas lemmatization preserved meaningful dictionary forms like "absolutely", "fantastic", "movie", and "terrible". It also correctly converted inflected words based on their grammatical role, for example "seen" → "see", "given" → "give", "was" → "be", and "worst" → "bad", while stemming either left some words unchanged or simply removed suffixes. Overall, the comparison demonstrates that although stemming is faster and slightly reduces vocabulary, POS-aware lemmatization produces cleaner and more semantically meaningful tokens, making it a better choice for most NLP tasks such as sentiment analysis and text classification.


---
## Viva Prep — Practice Questions

1. **What's the fundamental difference in approach between stemming and lemmatization?**
   Stemming applies fixed suffix-stripping rules with no real knowledge of the language; lemmatization looks words up against an actual vocabulary/morphological database to return a real dictionary base form.

2. **Why does WordNetLemmatizer need a part-of-speech tag to work correctly?**
   Because the correct base form of a word depends on its role in the sentence — e.g. "seen" only reduces to "see" if treated as a verb; treated as the default noun, WordNetLemmatizer has nothing to reduce it to and returns it unchanged.

3. **Give an example of over-stemming (or explain what it means).**
   Over-stemming is when a stemmer strips too aggressively and produces a form that isn't a real word, or incorrectly merges words that shouldn't be treated as the same (e.g. "studies" -> "studi").

4. **Why might a stemmer fail on an irregular verb like "outdid" or "given"?**
   Because stemmers only recognize suffix patterns, not irregular morphology — irregular forms don't follow a strippable-suffix rule, so the stemmer has no rule to apply and leaves the word unchanged.

5. **What's the practical trade-off between choosing stemming vs lemmatization for a real pipeline?**
   Stemming is faster and simpler but less accurate, and can produce non-words; lemmatization is more linguistically correct and produces real dictionary forms, but is slower and its accuracy depends on first getting POS tagging right.
